In [43]:
import glob
import torch
import sys
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from scipy.io import wavfile
from scipy import signal
from sklearn.decomposition import PCA

sys.path.append('/om2/user/bjmedina/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params

class AudioTextureEncoder(nn.Module):
    def __init__(self, statistics_dict, model_params, sr=20000, rms_level=0.04, duration=2.0, device='cuda'):
        super().__init__()
        self.sr = sr
        self.rms_level = rms_level
        self.duration = duration
        self.device = device

        self.flatten_stat_dict = FlattenStats(statistics_dict)
        coch_params, mod_params, octmod_params = model_params.update_param_dicts(duration)
        coch_params['rep_kwargs']['coch_filter_kwargs']['n'] = 30
        self.texture_model = TextureModel(coch_params, mod_params, octmod_params,
                                          statistics_dict=statistics_dict).to(device)

    def forward(self, filepath):
        """
        Accepts a string path to a .wav file, returns a flattened texture representation tensor.
        """
        original_sr, sound = wavfile.read(filepath)
        if sound.ndim > 1:
            sound = sound.mean(axis=1)
        sound = signal.resample_poly(sound, self.sr, original_sr, axis=0)
        sound = sound - np.mean(sound)
        sound = self.rms_level * sound / np.sqrt(np.mean(np.square(sound)))
        sound_tensor = torch.from_numpy(sound).float().unsqueeze(0).unsqueeze(0).to(self.device)

        with torch.no_grad():
            stats_dict = self.texture_model(sound_tensor)
            stats = self.flatten_stat_dict(stats_dict).squeeze(0)
        return stats

class PCASpace(nn.Module):
    def __init__(self, encoder, n_components=2, device='cpu'):
        super().__init__()
        self.encoder = encoder
        self.n_components = n_components
        self.device = device
        self.pca = None

    def fit(self, filepaths):
        """Fit PCA on a list of audio filepaths."""
        reps = []
        for path in filepaths:
            rep = self.encoder(path).detach().cpu().numpy()
            reps.append(rep)
        X = np.vstack(reps)
        self.pca = PCA(n_components=self.n_components)
        self.pca.fit(X)

    def transform(self, filepaths):
        """Project new filepaths into PCA space."""
        assert self.pca is not None, "You must call .fit() first"
        reps = []
        for path in filepaths:
            rep = self.encoder(path).detach().cpu().numpy()
            reps.append(rep)
        return self.pca.transform(np.vstack(reps))

    def fit_transform(self, filepaths):
        """Convenience method to fit and then project."""
        self.fit(filepaths)
        return self.transform(filepaths)

    def forward(self, filepath):
        """Encode + project a single file."""
        assert self.pca is not None, "Call fit() first."
        rep = self.encoder(filepath).detach().cpu().numpy().reshape(1, -1)
        proj = self.pca.transform(rep)
        return torch.tensor(proj, dtype=torch.float32).squeeze(0).to(self.device)

import os
import shutil
import json

def copy_selected_sounds_with_metadata(selected_paths, dest_folder, rename=True, metadata_filename="filenames.json"):
    """
    selected_paths: list of original .wav filepaths
    dest_folder: where to copy the sounds
    rename: if True, rename as sound_001.wav, etc.
    metadata_filename: name of the .json metadata file
    """
    os.makedirs(dest_folder, exist_ok=True)
    metadata = []

    for i, orig_path in enumerate(selected_paths):
        if rename:
            filename = f"mem_stim_{i}.wav"
        else:
            filename = os.path.basename(orig_path)

        stim_path = os.path.join(dest_folder, filename)
        shutil.copy2(orig_path, stim_path)

        entry = {
            "orig_path": orig_path,
            "stim_path": stim_path,
            "filename": filename,
            "onset": 0.0,
            "type": "texture"
        }
        metadata.append(entry)

    # Save metadata
    metadata_path = os.path.join(dest_folder, metadata_filename)
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Copied {len(selected_paths)} files to '{dest_folder}'")
    print(f"Metadata saved to '{metadata_path}'")

sounds_list_1 = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
sounds_list_2 = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p2/*wav")

texture_list = sounds_list_1 + sounds_list_2

mem_exp_atexts_p1_dislikes = [3, 6, 7, 10, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 25, 26, 28, 30, 34, 36, 38, 39, 41,
                             42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 56, 57, 58, 59]
mem_exp_atexts_p2_dislikes = []

In [44]:
mem_exp_atexts_p1_likes = []
for i in range(60):
    if not i in mem_exp_atexts_p1_dislikes:
        mem_exp_atexts_p1_likes.append(i)

In [45]:
mem_exp_atexts_p1_likes += [62, 64, 71, 72, 75, 76, 78, 85, 86, 93, 98, 100, 104,
                           108, 115, 130, 133, 136, 149, 154, 167, 170, 172]



In [46]:
mem_exp_atexts_p2_likes = [0, 1, 2, 3, 4, 7, 10, 15, 18, 20, 24, 36, 43, 44, 48, 
                           61, 65, 73, 74, 75, 76, 84, 97, 98, 99, 122, 129, 131, 132,
                           134, 135, 139, 151, 160, 173 ]

print(len(mem_exp_atexts_p1_likes) + len(mem_exp_atexts_p2_likes))

80


In [57]:
p1_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_{}.wav"
p2_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p2/mem_stim_{}.wav"
texture_list = [p1_path.format(i) for i in mem_exp_atexts_p1_likes  if p1_path.format(i) in sounds_list_1 ]
texture_list += [p2_path.format(i) for i in mem_exp_atexts_p2_likes  if p2_path.format(i) in sounds_list_2 ]
print(f"We have {len(texture_list)} textures")
print(*texture_list, sep='\n')

We have 80 textures
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_0.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_1.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_2.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_4.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_5.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_8.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_9.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_11.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_15.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_22.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_24.wav
/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_27.wav
/mindhive/mcder

In [51]:
f"/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/mem_stim_{34}.wav" in sounds_list_1

True

In [48]:
# get the texture model
texture_model = AudioTextureEncoder(
    statistics_dict=statistics_dict,
    model_params=model_params,
    sr=20000,
    rms_level=0.04,
    duration=2.0,
    device='cuda'
)

In [3]:
from torch.nn.functional import pairwise_distance

def greedy_k_center(reps, k=80):
    selected = [0]  # start from the first one
    distances = torch.cdist(reps, reps, p=2)

    for _ in range(1, k):
        dists_to_selected = distances[:, selected]
        min_dists = dists_to_selected.min(dim=1)[0]
        next_idx = min_dists.argmax().item()
        selected.append(next_idx)
    
    return selected

def maximize_feature_variance(reps, k=80):
    """
    reps: torch.Tensor of shape (N, D)
    returns: list of indices of selected points
    """
    selected = []
    remaining = list(range(len(reps)))

    # Start with the point that has the highest L2 norm (furthest from origin)
    start_idx = torch.norm(reps, dim=1).argmax().item()
    selected.append(start_idx)
    remaining.remove(start_idx)

    while len(selected) < k:
        best_score = -float('inf')
        best_idx = None

        for idx in remaining:
            candidate = selected + [idx]
            subset = reps[candidate]
            var = subset.var(dim=0).sum().item()  # total variance across all features
            if var > best_score:
                best_score = var
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return selected

In [4]:
%%time
# get texture statisitcs
reps = []
valid_paths = []
for path in texture_list:
    try:
        rep = texture_model(path)
        reps.append(rep.cpu())
        valid_paths.append(path)
    except Exception as e:
        print(f"Skipping {path} due to error: {e}")

reps = torch.stack(reps)  # shape: (N, D)

CPU times: user 3.72 s, sys: 249 ms, total: 3.97 s
Wall time: 4.08 s


In [5]:
greedy_indices  = greedy_k_center(reps, k=80)
max_var_indices = maximize_feature_variance(reps, k=80)

In [6]:
max_var_paths = [texture_list[i] for i in max_var_indices]

copy_selected_sounds_with_metadata(
    max_var_paths,
    dest_folder=f'/mindhive/mcdermott/www/mturk_stimuli/bjmedina/max_var_textures_{len(max_var_indices)}',
    rename=True,
    metadata_filename='filenames.json'
)

Copied 80 files to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/max_var_textures_80'
Metadata saved to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/max_var_textures_80/filenames.json'


In [7]:
greedy_paths = [texture_list[i] for i in greedy_indices]

copy_selected_sounds_with_metadata(
    greedy_paths,
    dest_folder=f'/mindhive/mcdermott/www/mturk_stimuli/bjmedina/greedy_textures_{len(max_var_indices)}',
    rename=True,
    metadata_filename='filenames.json'
)

Copied 80 files to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/greedy_textures_80'
Metadata saved to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/greedy_textures_80/filenames.json'


In [58]:

copy_selected_sounds_with_metadata(
    texture_list,
    dest_folder=f'/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025',
    rename=True,
    metadata_filename='filenames.json'
)

Copied 80 files to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025'
Metadata saved to '/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/filenames.json'
